In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# ==========================================
# 1. LOAD THE DATA (SIGNAL + BMI)
# ==========================================
print("Loading Deep Learning Dataset (Signal + BMI)...")
X_signals = np.load("X_signals.npy")  # Shape: (2150, 100, 2)
X_bmi = np.load("X_bmi.npy")          # Shape: (2150,)
y_glucose = np.load("y_glucose.npy")  # Shape: (2150,)

X_sig_train, X_sig_test, X_bmi_train, X_bmi_test, y_train, y_test = train_test_split(
    X_signals, X_bmi, y_glucose, test_size=0.2, random_state=42
)

Loading Deep Learning Dataset (Signal + BMI)...


In [3]:
# ==========================================
# 2. BUILD THE HYBRID CNN-BiLSTM ARCHITECTURE
# ==========================================

# --- HEAD 1: The Spatio-Temporal Signal Processor ---
input_signal = layers.Input(shape=(100, 2), name="signal_input")

# Step A: CNN extracts the local geometric shapes (peaks, slopes)
x = layers.Conv1D(filters=64, kernel_size=5, activation='relu', padding='same')(input_signal)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling1D(pool_size=2)(x) # Output shape is now (50, 64)

x = layers.Conv1D(filters=128, kernel_size=3, activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling1D(pool_size=2)(x) # Output shape is now (25, 128)

# Step B: The BiLSTM reads the sequence to track the "Flow of Time"
# return_sequences=True passes the full time-flow to the next layer
x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
x = layers.Dropout(0.3)(x)

# return_sequences=False squashes the final temporal understanding into a flat array
x = layers.Bidirectional(layers.LSTM(32, return_sequences=False))(x)

# --- HEAD 2: The BMI Processor ---
input_bmi = layers.Input(shape=(1,), name="bmi_input")
y_branch = layers.Dense(16, activation='relu')(input_bmi)
y_branch = layers.BatchNormalization()(y_branch)

# --- THE MERGE: Combine Time, Shape, and BMI ---
merged = layers.Concatenate()([x, y_branch])

# Final reasoning layers
z = layers.Dense(64, activation='relu')(merged)
z = layers.Dropout(0.3)(z)
z = layers.Dense(32, activation='relu')(z)

output = layers.Dense(1, activation='linear', name="glucose_output")(z)

# Compile the Hybrid Model
model = Model(inputs=[input_signal, input_bmi], outputs=output)

# BiLSTMs require a very careful, steady learning rate
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ signal_input        │ (None, 100, 2)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 100, 64)   │        704 │ signal_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 100, 64)   │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 50, 64)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 50, 128)   │     24,704 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 128)   │        512 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 25, 128)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 25, 128)   │     98,816 │ max_pooling1d_1[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bmi_input           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 25, 128)   │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │         32 │ bmi_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 64)        │     41,216 │ dropout[0][0]     │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16)        │         64 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 80)        │          0 │ bidirectional_1[… │
│ (Concatenate)       │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      5,184 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ glucose_output      │ (None, 1)         │         33 │ dense_2[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 173,601 (678.13 KB)

 Trainable params: 173,185 (676.50 KB)

 Non-trainable params: 416 (1.62 KB)

In [4]:
# ==========================================
# 3. TRAIN THE MODEL
# ==========================================
print("\nTraining the CNN-BiLSTM Hybrid Model...")

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=0.00001)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)

history = model.fit(
    x=[X_sig_train, X_bmi_train], 
    y=y_train,
    validation_data=([X_sig_test, X_bmi_test], y_test),
    epochs=150,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


Training the CNN-BiLSTM Hybrid Model...
Epoch 1/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 12s 34ms/step - loss: 10613.8242 - mae: 95.2084 - val_loss: 3773.2314 - val_mae: 50.6200 - learning_rate: 0.0010
Epoch 2/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 1917.1116 - mae: 34.3488 - val_loss: 4487.0034 - val_mae: 54.0734 - learning_rate: 0.0010
Epoch 3/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 1515.6309 - mae: 30.7803 - val_loss: 13710.9580 - val_mae: 111.7182 - learning_rate: 0.0010
Epoch 4/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 1439.8943 - mae: 30.0950 - val_loss: 13792.9189 - val_mae: 112.0599 - learning_rate: 0.0010
Epoch 5/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 1461.3369 - mae: 30.9067 - val_loss: 8387.4531 - val_mae: 75.5396 - learning_rate: 0.0010
Epoch 6/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 1450.3208 - mae: 30.6447 - val_loss: 11546.4277 - val_mae: 97.5205 - learning_rate: 0.0010
Epoch 7/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step 

In [5]:
# ==========================================
# 4. EVALUATE
# ==========================================
print("\n==================================================")
print("🏆 HYBRID CNN-BiLSTM RESULTS")
print("==================================================")

predictions = model.predict([X_sig_test, X_bmi_test]).flatten()
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"➡️ Mean Absolute Error (MAE): {mae:.2f} mg/dL")
print(f"➡️ Root Mean Squared Error (RMSE): {rmse:.2f} mg/dL")
print(f"➡️ R-Squared (R2):            {r2:.2f}")


🏆 HYBRID CNN-BiLSTM RESULTS
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step
➡️ Mean Absolute Error (MAE): 23.41 mg/dL
➡️ Root Mean Squared Error (RMSE): 30.19 mg/dL
➡️ R-Squared (R2):            0.26


In [6]:
model.save("glucose_hybrid_bilstm.h5")
print("\nModel saved successfully as 'glucose_hybrid_bilstm.h5'")


Model saved successfully as 'glucose_hybrid_bilstm.h5'
